# Lab 06 External V2 — 08 Governance: GRANT, RLS and CLS

**Dataset:** Synthea Healthcare  
**Architecture:** Unity Catalog external Gold tables  
**Purpose:** Apply and validate governance on the final External V2 Gold model.

This notebook demonstrates:

- **GRANTs** for Unity Catalog object access;
- **Row-Level Security (RLS)** on `fact_encounters`, based on `organization_id`;
- **Column-Level Security (CLS)** on `dim_patient`, masking:
  - `ssn`
  - `first_name`
  - `last_name`
  - `address`

The notebook temporarily restricts the invoking user to prove the policies work,
then restores full access for that user while leaving the policies attached.

> Governance is intentionally outside the recurring seven-task Gold Job.
> The Gold Job owns data processing; this notebook owns security-policy setup and validation.

> Run this notebook only after the External V2 Gold tables have been created and registered.


## 1. Parameters

Only the Unity Catalog namespace is required. Governance policies are metadata
objects and do not modify the shared external ADLS Delta files.


In [0]:
def ensure_text_widget(name: str, default: str, label: str) -> None:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default, label)


def ensure_dropdown_widget(
    name: str,
    default: str,
    choices: list[str],
    label: str,
) -> None:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.dropdown(name, default, choices, label)


ensure_text_widget("catalog", "dbr_dev", "01 Catalog")
ensure_text_widget(
    "target_schema",
    "parvinbadalov_lab06_ext",
    "02 External V2 target schema",
)
ensure_dropdown_widget(
    "run_demo",
    "true",
    ["true", "false"],
    "03 Run restricted/masked demo",
)

catalog = dbutils.widgets.get("catalog").strip()
target_schema = dbutils.widgets.get("target_schema").strip()
run_demo = dbutils.widgets.get("run_demo").strip().lower() == "true"

target_schema_fqn = f"{catalog}.{target_schema}"

print(f"Catalog       : {catalog}")
print(f"Target schema : {target_schema}")
print(f"Run demo      : {run_demo}")


## 2. Resolve External V2 governance objects

The policies are attached to the registered Unity Catalog tables, not directly
to the ADLS paths.


In [0]:
from pyspark.sql import functions as F

fact_encounters = f"{target_schema_fqn}.fact_encounters"
dim_patient = f"{target_schema_fqn}.dim_patient"

user_org_access = f"{target_schema_fqn}.lab06_user_organization_access"
privileged_users = f"{target_schema_fqn}.lab06_patient_data_privileged_users"

org_filter_function = f"{target_schema_fqn}.lab06_org_row_filter"
mask_function = f"{target_schema_fqn}.lab06_mask_sensitive_string"

for table_name in [fact_encounters, dim_patient]:
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Missing governance target: {table_name}. "
            "Run the External V2 Gold Job first."
        )

fact_columns = set(spark.table(fact_encounters).columns)
patient_columns = set(spark.table(dim_patient).columns)

if "organization_id" not in fact_columns:
    raise RuntimeError(
        f"{fact_encounters} must contain organization_id for the RLS policy."
    )

SENSITIVE_COLUMNS = ["ssn", "first_name", "last_name", "address"]
missing_sensitive = sorted(set(SENSITIVE_COLUMNS) - patient_columns)

if missing_sensitive:
    raise RuntimeError(
        "dim_patient is missing required CLS columns: "
        + ", ".join(missing_sensitive)
    )

print(f"RLS target : {fact_encounters}")
print(f"CLS target : {dim_patient}")


## 3. Resolve the invoking user

`SESSION_USER()` is evaluated by the row-filter and masking functions so policy
behavior follows the identity of the user running a query.


In [0]:
session_user = spark.sql(
    "SELECT SESSION_USER() AS username"
).first()["username"]


def quote_principal(value: str) -> str:
    return "`" + value.replace("`", "``") + "`"


principal_sql = quote_principal(session_user)
escaped_user = session_user.replace("'", "''")

print(f"Session user: {session_user}")


## 4. Capture the unrestricted baseline

The demo uses the organization with the largest number of encounters as the
temporary RLS scope.


In [0]:
baseline_encounter_count = spark.table(fact_encounters).count()

demo_org_row = (
    spark.table(fact_encounters)
    .filter(F.col("organization_id").isNotNull())
    .groupBy("organization_id")
    .count()
    .orderBy(F.desc("count"))
    .first()
)

if demo_org_row is None:
    raise RuntimeError("No organization_id is available for the RLS demo.")

demo_organization_id = str(demo_org_row["organization_id"])
demo_organization_count = int(demo_org_row["count"])

print(f"Baseline rows          : {baseline_encounter_count:,}")
print(f"Demo organization      : {demo_organization_id}")
print(f"Demo organization rows : {demo_organization_count:,}")


## 5. Create governance mapping tables

These are small Unity Catalog Delta tables used by the policy functions.

- `lab06_user_organization_access` controls row visibility.
- `lab06_patient_data_privileged_users` controls access to unmasked patient values.

They are metadata/control tables; the business Gold tables remain external.


In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {user_org_access} (
    username STRING NOT NULL,
    organization_id STRING NOT NULL,
    access_reason STRING,
    updated_at TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {privileged_users} (
    username STRING NOT NULL,
    access_reason STRING,
    updated_at TIMESTAMP
)
USING DELTA
""")

print("Governance mapping tables are ready.")


## 6. Create the organization row-filter function

Policy behavior is **default deny**:

- matching organization → visible;
- `*` mapping → all organizations;
- no mapping → no encounter rows.


In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {org_filter_function} (
    row_organization_id STRING
)
RETURN EXISTS (
    SELECT 1
    FROM {user_org_access} a
    WHERE a.username = SESSION_USER()
      AND (
          a.organization_id = row_organization_id
          OR a.organization_id = '*'
      )
)
""")

print(f"Created: {org_filter_function}")


## 7. Prepare restricted RLS demo access

When `run_demo=true`, the invoking user temporarily receives access to only one
organization.


In [0]:
if run_demo:
    escaped_org = demo_organization_id.replace("'", "''")

    spark.sql(
        f"DELETE FROM {user_org_access} "
        f"WHERE username = '{escaped_user}'"
    )

    spark.sql(f"""
    INSERT INTO {user_org_access}
    VALUES (
        '{escaped_user}',
        '{escaped_org}',
        'Lab 06 External V2 restricted RLS demo',
        CURRENT_TIMESTAMP()
    )
    """)

    display(
        spark.table(user_org_access)
        .filter(F.col("username") == session_user)
    )
else:
    print("Restricted RLS demo skipped.")


## 8. Apply RLS to `fact_encounters`


In [0]:
spark.sql(f"""
ALTER TABLE {fact_encounters}
SET ROW FILTER {org_filter_function}
ON (organization_id)
""")

print("RLS policy attached.")


## 9. Validate restricted RLS

A successful demo shows that the visible encounter count is reduced from the
full baseline to exactly the selected organization's rows.


In [0]:
if run_demo:
    restricted_encounter_count = spark.table(fact_encounters).count()

    rls_restricted_status = (
        "PASS"
        if restricted_encounter_count == demo_organization_count
        else "FAIL"
    )

    display(
        spark.createDataFrame(
            [(
                baseline_encounter_count,
                demo_organization_count,
                restricted_encounter_count,
                rls_restricted_status,
            )],
            [
                "baseline_rows",
                "expected_restricted_rows",
                "actual_restricted_rows",
                "status",
            ],
        )
    )

    if rls_restricted_status == "FAIL":
        raise RuntimeError("Restricted RLS validation failed.")
else:
    rls_restricted_status = "SKIPPED"


## 10. Restore full RLS access for the invoking user

The row filter remains attached. The current user receives a wildcard mapping so
normal development and future Job runs can continue with the complete dataset.

Unmapped users remain default-denied.


In [0]:
spark.sql(
    f"DELETE FROM {user_org_access} "
    f"WHERE username = '{escaped_user}'"
)

spark.sql(f"""
INSERT INTO {user_org_access}
VALUES (
    '{escaped_user}',
    '*',
    'Lab 06 External V2 full organization access',
    CURRENT_TIMESTAMP()
)
""")

restored_encounter_count = spark.table(fact_encounters).count()

rls_restored_status = (
    "PASS"
    if restored_encounter_count == baseline_encounter_count
    else "FAIL"
)

display(
    spark.createDataFrame(
        [(
            baseline_encounter_count,
            restored_encounter_count,
            rls_restored_status,
        )],
        ["baseline_rows", "restored_rows", "status"],
    )
)

if rls_restored_status == "FAIL":
    raise RuntimeError("RLS full-access restoration failed.")


## 11. Create the patient-data masking function

Users listed in `lab06_patient_data_privileged_users` see the original value.
Other users see `***MASKED***` for non-null values.


In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {mask_function} (
    value STRING
)
RETURN CASE
    WHEN EXISTS (
        SELECT 1
        FROM {privileged_users} p
        WHERE p.username = SESSION_USER()
    )
    THEN value
    WHEN value IS NULL THEN NULL
    ELSE '***MASKED***'
END
""")

print(f"Created: {mask_function}")


## 12. Apply CLS masks to `dim_patient`


In [0]:
for column_name in SENSITIVE_COLUMNS:
    spark.sql(f"""
    ALTER TABLE {dim_patient}
    ALTER COLUMN {column_name}
    SET MASK {mask_function}
    """)

print("Masked columns: " + ", ".join(SENSITIVE_COLUMNS))


## 13. Validate masked CLS behavior

When `run_demo=true`, the invoking user is temporarily removed from the
privileged mapping and the notebook confirms that sensitive values are masked.


In [0]:
if run_demo:
    spark.sql(
        f"DELETE FROM {privileged_users} "
        f"WHERE username = '{escaped_user}'"
    )

    masked_sample = (
        spark.table(dim_patient)
        .select(
            "patient_id",
            "ssn",
            "first_name",
            "last_name",
            "address",
        )
        .filter(F.col("ssn").isNotNull())
        .limit(5)
    )

    display(masked_sample)

    unmasked_non_null_ssn = (
        masked_sample
        .filter(
            F.col("ssn").isNotNull()
            & (F.col("ssn") != "***MASKED***")
        )
        .count()
    )

    cls_masked_status = (
        "PASS" if unmasked_non_null_ssn == 0 else "FAIL"
    )

    if cls_masked_status == "FAIL":
        raise RuntimeError("CLS masked-value validation failed.")
else:
    cls_masked_status = "SKIPPED"


## 14. Restore privileged CLS access for the invoking user

The masks remain attached. The current user is added to the privileged mapping
so normal development remains readable, while non-privileged users continue to
receive masked values.


In [0]:
spark.sql(
    f"DELETE FROM {privileged_users} "
    f"WHERE username = '{escaped_user}'"
)

spark.sql(f"""
INSERT INTO {privileged_users}
VALUES (
    '{escaped_user}',
    'Lab 06 External V2 patient-data privileged access',
    CURRENT_TIMESTAMP()
)
""")

privileged_sample = (
    spark.table(dim_patient)
    .select(
        "patient_id",
        "ssn",
        "first_name",
        "last_name",
        "address",
    )
    .filter(F.col("ssn").isNotNull())
    .limit(5)
)

display(privileged_sample)

cls_restored_status = (
    "PASS"
    if privileged_sample
       .filter(F.col("ssn") == "***MASKED***")
       .count() == 0
    else "FAIL"
)

if cls_restored_status == "FAIL":
    raise RuntimeError("CLS restoration validation failed.")


## 15. Apply and verify GRANTs

For a personal validation run, the session user is used as the principal.
In shared environments, account-level groups are preferable.

The user executing this notebook must have sufficient Unity Catalog privileges
to create functions and alter the governed tables.


In [0]:
grant_statements = [
    (
        f"GRANT USE CATALOG ON CATALOG {catalog} "
        f"TO {principal_sql}"
    ),
    (
        f"GRANT USE SCHEMA ON SCHEMA {target_schema_fqn} "
        f"TO {principal_sql}"
    ),
    (
        f"GRANT SELECT ON TABLE {fact_encounters} "
        f"TO {principal_sql}"
    ),
    (
        f"GRANT SELECT ON TABLE {dim_patient} "
        f"TO {principal_sql}"
    ),
]

for statement in grant_statements:
    spark.sql(statement)

display(
    spark.sql(
        f"SHOW GRANTS ON TABLE {fact_encounters}"
    )
)

print(f"GRANTs verified for: {session_user}")


## 16. Final governance validation


In [0]:
final_checks = [
    ("RLS restricted demo", rls_restricted_status),
    ("RLS full-access restoration", rls_restored_status),
    ("CLS masked demo", cls_masked_status),
    ("CLS privileged restoration", cls_restored_status),
]

final_validation_df = spark.createDataFrame(
    final_checks,
    ["governance_check", "status"],
)

display(final_validation_df)

failed_checks = [
    check_name
    for check_name, status in final_checks
    if status == "FAIL"
]

if failed_checks:
    raise RuntimeError(
        "External V2 governance validation failed: "
        + ", ".join(failed_checks)
    )


## 17. Final policy state

```text
dbr_dev.<target_schema>.fact_encounters
  └── RLS attached on organization_id
      ├── mapped organization → matching rows
      ├── "*" mapping         → all organizations
      └── unmapped user       → no rows

dbr_dev.<target_schema>.dim_patient
  ├── ssn        → masked for non-privileged users
  ├── first_name → masked for non-privileged users
  ├── last_name  → masked for non-privileged users
  └── address    → masked for non-privileged users
```

The policies remain attached after the notebook completes.

The invoking user is deliberately restored to:

- `*` organization access;
- privileged patient-data access.

This prevents the security demo from breaking future Gold Job runs executed
under the same identity.


In [0]:
print("LAB 06 EXTERNAL V2 — GOVERNANCE COMPLETE")
print(f"RLS target : {fact_encounters}")
print(f"CLS target : {dim_patient}")
print(f"Principal  : {session_user}")
print("RLS and CLS remain enabled.")
print("Current user restored to full / privileged access.")


## Evidence to capture

For GitHub, capture two screenshots while `run_demo=true`:

1. **RLS:** the validation table showing:
   - full baseline rows;
   - expected restricted rows;
   - actual restricted rows;
   - `PASS`.

2. **CLS:** the `dim_patient` sample showing `***MASKED***` in the sensitive
   columns.

Recommended filenames:

```text
09_governance_rls.png
10_governance_cls.png
```

These two screenshots are enough to prove row-level and column-level security.


## Optional rollback

Do not run rollback during the normal Lab 06 flow.

Rollback SQL is provided in:

```text
sql/governance_policies_external.sql
```

Always remove the row filter / column masks before dropping the UDFs referenced
by those policies.
